# CARec encoder and dimension study

- Load the saved Amazon Electronics catalogue, interaction partitions, and ID maps.
- Use the `NI-PML benchmarks` kernel from the project `.venv`.
- Data exploration, text preparation, sampling, splitting, and saving belong in [data_exploration.ipynb](data_exploration.ipynb).


In [24]:
from pathlib import Path
import gzip
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
PREPARED_DIR = PROJECT_ROOT / "data/processed/electronics_carec_035_seed2023"
print(f"Prepared dataset: {PREPARED_DIR}")


Prepared dataset: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023


## Load products and ID maps

- Read one product record per JSONL line.
- Load user and product ID maps, keeping their IDs as strings.


In [25]:
items_path = PREPARED_DIR / "items.jsonl.gz"
product_records = []

with gzip.open(items_path, "rt", encoding="utf-8") as source:
    for line in source:
        product = json.loads(line)
        product_records.append(product)

items = pd.DataFrame(product_records)
del product_records

user_map_path = PREPARED_DIR / "user_ids.tsv.gz"
user_map = pd.read_csv(user_map_path, sep="\t", dtype={"user_id": str})

item_map_path = PREPARED_DIR / "item_ids.tsv.gz"
item_map = pd.read_csv(item_map_path, sep="\t", dtype={"item_id": str})


## Load interaction partitions

- Load train, validation, test, and cold interactions.
- Keep user and product IDs as strings and preserve float precision.


In [26]:
partition_names = ("train", "valid", "test", "cold")

interaction_types = {
    "user_id:token": str,
    "item_id:token": str,
}
partitions = {}

for name in partition_names:
    partition_path = PREPARED_DIR / f"{name}.tsv.gz"
    frame = pd.read_csv(
        partition_path,
        sep="\t",
        dtype=interaction_types,
        float_precision="round_trip",
    )
    partitions[name] = frame

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


## Show the loaded data

- Print product, user, and partition counts.
- Show two product texts.


In [27]:
print(f"Loaded {len(items):,} products and {len(user_map):,} users")

for name, frame in partitions.items():
    print(f"{name}: {len(frame):,} interactions")

text_preview = items[["item_id:token", "text"]].head(2)
display(text_preview)


Loaded 34,220 products and 81,520 users
train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


,item_id:token,text
0,0545105668,Elephant Run Visit Amazon's Roland Smith Page ...
1,0789743035,My MacBook Visit Amazon's John Ray Page Compu...


## Encoder candidates

- `text` is the saved product text from `items["text"]`; `instruction` describes the recommendation task. The pilot instruction below is provisional.
- Run the three instruction models with and without instructions: plain `text` for Qwen/E5-instruct, an empty prompt for INSTRUCTOR. Pilot these variants before bulk generation.
- E5-base's `query: ` prefix is required formatting, not a task instruction. `\n` below means a newline.

| Model | Instruction-based | Input format |
| --- | --- | --- |
| [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | No | `text` |
| [all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | No | `text` |
| [e5-base-v2](https://huggingface.co/intfloat/e5-base-v2) | No | `query: {text}` |
| [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) | Yes | `Instruct: {instruction}\nQuery:{text}` |
| [multilingual-e5-large-instruct](https://huggingface.co/intfloat/multilingual-e5-large-instruct) | Yes | `Instruct: {instruction}\nQuery: {text}` |
| [instructor-base](https://huggingface.co/hkunlp/instructor-base) | Yes | `text` with `prompt=instruction`, excluding the prompt from pooling |


## One-batch encoder pilots

### MiniLM-L6-v2

- 22.7 million parameters, 6 Transformer layers.
- Output dimension: 384.
- No task instruction or prefix; input is the saved product text.
- Inputs longer than 256 tokens are truncated. [Model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).


In [44]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

PILOT_INSTRUCTION = "Encode this sentence for reccomendation purposes"

device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 4

print("running on", device)
print()
sample_texts = items["text"].head(batch_size).tolist()
print("Sample text:", sample_texts[0])

running on mps

Sample text: Elephant Run Visit Amazon's Roland Smith Page  eBook Readers & Accessories eBook Readers.


In [ ]:
model_id = "sentence-transformers/all-MiniLM-L6-v2"
minilm = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
)
minilm_embeddings = minilm.encode(
    sample_texts,
    batch_size=batch_size,
    prompt="",
    normalize_embeddings=True,
    show_progress_bar=False,
)
print(minilm_embeddings.shape)


(4, 384)


### MPNet-base-v2

- 109.5 million parameters, 12 Transformer layers.
- Output dimension: 768.
- No task instruction or prefix; input is the saved product text.
- Inputs longer than 384 tokens are truncated. [Model card](https://huggingface.co/sentence-transformers/all-mpnet-base-v2).


In [ ]:
model_id = "sentence-transformers/all-mpnet-base-v2"
mpnet = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
)
mpnet_embeddings = mpnet.encode(
    sample_texts,
    batch_size=batch_size,
    prompt="",
    normalize_embeddings=True,
    show_progress_bar=False,
)
print(mpnet_embeddings.shape)


(4, 768)


### E5-base-v2

- About 110 million parameters, 12 Transformer layers.
- Output dimension: 768.
- No task instruction. Prefix each product text with `query: ` when using embeddings as features.
- English texts only; inputs longer than 512 tokens are truncated. [Model card](https://huggingface.co/intfloat/e5-base-v2).


In [ ]:
model_id = "intfloat/e5-base-v2"
print("query: " + sample_texts[0])

e5base = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
)
e5base_embeddings = e5base.encode(
    sample_texts,
    batch_size=batch_size,
    prompt="query: ",
    normalize_embeddings=True,
    show_progress_bar=False,
)
print(e5base_embeddings.shape)


### Qwen3-Embedding-0.6B

- About 0.6 billion parameters, 28 Transformer layers.
- Output dimension: 1024; input limit: 32,768 tokens.
- Instruction-based. Use `Instruct: {instruction}\nQuery:{text}`.
- Use eager attention on this MPS setup; SDPA returned non-finite vectors. [Model card](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B).


In [ ]:
qwen_prompt = f"Instruct: {PILOT_INSTRUCTION}\nQuery:"

print("Without instruction:")
print(sample_texts[0])
print("\nWith instruction:")
print(qwen_prompt + sample_texts[0])


In [ ]:
model_id = "Qwen/Qwen3-Embedding-0.6B"
qwen = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
    model_kwargs={"attn_implementation": "eager", "dtype": torch.float32},
    tokenizer_kwargs={"padding_side": "left"},
)

qwen_embeddings = {}
for mode, prompt in {
    "without_instruction": "",
    "with_instruction": qwen_prompt,
}.items():
    qwen_embeddings[mode] = qwen.encode(
        sample_texts,
        batch_size=batch_size,
        prompt=prompt,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    assert np.isfinite(qwen_embeddings[mode]).all(), "Non-finite embeddings"
    print(mode, qwen_embeddings[mode].shape)


### Multilingual-E5-large-instruct

- About 0.6 billion parameters, 24 Transformer layers.
- Output dimension: 1024; input limit: 512 tokens.
- Instruction-based. Use `Instruct: {instruction}\nQuery: {text}`.
- Pilot both with and without the shared instruction. [Model card](https://huggingface.co/intfloat/multilingual-e5-large-instruct).


In [ ]:
e5instruct_prompt = f"Instruct: {PILOT_INSTRUCTION}\nQuery: "

print("Without instruction:")
print(sample_texts[0])
print("\nWith instruction:")
print(e5instruct_prompt + sample_texts[0])


In [ ]:
model_id = "intfloat/multilingual-e5-large-instruct"
e5instruct = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
)

e5instruct_embeddings = {}
for mode, prompt in {
    "without_instruction": "",
    "with_instruction": e5instruct_prompt,
}.items():
    e5instruct_embeddings[mode] = e5instruct.encode(
        sample_texts,
        batch_size=batch_size,
        prompt=prompt,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    print(mode, e5instruct_embeddings[mode].shape)


### INSTRUCTOR-base

- About 110 million parameters, based on a T5 encoder.
- Output dimension: 768; input limit: 512 tokens.
- Instruction-based. Pass the shared instruction through `prompt=` and exclude its tokens from pooling.
- Pilot both with and without the instruction. [Model card](https://huggingface.co/hkunlp/instructor-base).


In [ ]:
instructor_prompt = PILOT_INSTRUCTION + " "

print("Without instruction:")
print(sample_texts[0])
print("\nWith instruction:")
print(instructor_prompt + sample_texts[0])


In [ ]:
model_id = "hkunlp/instructor-base"
instructor = SentenceTransformer(
    model_id,
    device=device,
    cache_folder=str(PROJECT_ROOT / ".cache/huggingface/hub"),
)
instructor.set_pooling_include_prompt(False)

instructor_embeddings = {}
for mode, prompt in {
    "without_instruction": "",
    "with_instruction": instructor_prompt,
}.items():
    instructor_embeddings[mode] = instructor.encode(
        sample_texts,
        batch_size=batch_size,
        prompt=prompt,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    assert np.isfinite(instructor_embeddings[mode]).all(), "Non-finite embeddings"
    print(mode, instructor_embeddings[mode].shape)


## Pilot results

- Show the output shapes from the pilot cells above. Run all six sections first.


In [ ]:
pilot_shapes = {
    "MiniLM": minilm_embeddings.shape,
    "MPNet": mpnet_embeddings.shape,
    "E5-base": e5base_embeddings.shape,
}
for model_name, embeddings in {
    "Qwen": qwen_embeddings,
    "E5-instruct": e5instruct_embeddings,
    "INSTRUCTOR": instructor_embeddings,
}.items():
    for mode, vectors in embeddings.items():
        pilot_shapes[f"{model_name}/{mode}"] = vectors.shape

display(pd.Series(pilot_shapes, name="shape"))
